# ARIA · Kaggle Training Notebook

**Meta PyTorch OpenEnv Hackathon 2026** — TRL GRPO fine-tune of Qwen 2.5 0.5B-Instruct on the ARIA env.

## One-time Kaggle setup (do these BEFORE running cells)

1. **Sign in** at kaggle.com (Google or email — both work)
2. **Verify your phone number** at kaggle.com/settings → Account → Phone Verification. Required by Kaggle for GPU access. ~2 minutes.
3. In this notebook's right sidebar:
   - **Settings → Accelerator** → choose **`GPU T4 x2`** (or `GPU P100`)
   - **Settings → Internet** → **On** (required to clone the repo + download the model)
   - **Settings → Persistence** → **`Files only`** (so checkpoints survive restarts)
4. Click **Save Version** (top-right) once → this commits the notebook so you can resume if a session times out
5. Run cells top-to-bottom.

## Pick your training mode (set in Cell 2)

| Mode | Steps | Generations | Per-run | Total | When to use |
|---|---|---|---|---|---|
| **`quick`** | 200 | 2 | ~20 min | **~75 min** | Smoke-test the pipeline; format-learning visible; ablation flat |
| **`detailed`** ← default | 500 | 4 | ~50 min | **~3 hours** | Real GRPO learning beyond the format floor; tighter ablation signal |
| **`deep`** | 1000 | 4 | ~95 min | **~5 hours** | Strongest training curve; needs Kaggle's full 9h session |

**Recommended for the hackathon submission: `detailed`.**

## What this notebook does

1. Sanity check the GPU + Internet
2. Install deps + clone the latest ARIA repo from GitHub
3. Smoke-test the env
4. **Run A**: train with full 6-dim reward
5. **Run B**: train with `relationship_health` ablated (same step budget for a fair comparison)
6. Plot the two reward curves on the same axes
7. Run eval over all 6 scenario categories × 3 difficulties × 5 seeds
8. Render the per-category leaderboard plot

All outputs land in `/kaggle/working/` and are downloadable from the notebook's **Output** tab when the session ends.


## 0 · Pre-flight checks

If GPU is missing → Settings → Accelerator → `GPU T4 x2`. Restart kernel after changing.
If Internet is off → Settings → Internet → On. No restart needed.

Run **Cell 2** (Training Mode) FIRST — it sets env vars used by Run A and Run B.


In [ ]:
# === TRAINING MODE — pick one ============================================
# Sets the GRPO hyperparameters used by Cells 8 + 10 (Run A and Run B).
# Both runs always use the SAME hyperparameters — that's what makes the
# ablation comparison fair.
TRAIN_MODE = 'detailed'   # 'quick' | 'detailed' | 'deep'

MODES = {
    'quick':    {'steps': 200,  'num_generations': 2, 'max_completion_len': 64,  'max_prompt_len': 768,  'note': '~20 min per run'},
    'detailed': {'steps': 500,  'num_generations': 4, 'max_completion_len': 128, 'max_prompt_len': 1024, 'note': '~50 min per run'},
    'deep':     {'steps': 1000, 'num_generations': 4, 'max_completion_len': 128, 'max_prompt_len': 1024, 'note': '~95 min per run'},
}
assert TRAIN_MODE in MODES, f'TRAIN_MODE must be one of {list(MODES)}'
M = MODES[TRAIN_MODE]
import os
os.environ['ARIA_TRAIN_STEPS']    = str(M['steps'])
os.environ['ARIA_TRAIN_NGEN']     = str(M['num_generations'])
os.environ['ARIA_TRAIN_MAXCOMP']  = str(M['max_completion_len'])
os.environ['ARIA_TRAIN_MAXPROMPT']= str(M['max_prompt_len'])
print(f"=== TRAIN_MODE: {TRAIN_MODE} ({M['note']}) ===")
for k, v in M.items():
    print(f'  {k:<22}{v}')


In [ ]:
# GPU + Internet sanity check
import subprocess, urllib.request, sys

print('=== GPU ===')
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], text=True))
except Exception as e:
    print(f'⚠ no GPU detected: {e}')
    print('  Fix: Settings → Accelerator → GPU T4 x2, then Run All again.')

print('=== Internet ===')
try:
    urllib.request.urlopen('https://github.com', timeout=5)
    print('✓ github.com reachable')
except Exception as e:
    print(f'⚠ no internet: {e}')
    print('  Fix: Settings → Internet → On (then Run All again).')

print('=== Python ===')
print(sys.version.split()[0])

# Repo HEAD — confirms you have the latest detailed-mode notebook.
import os
if os.path.exists('/kaggle/working/Aria/.git'):
    try:
        head = subprocess.check_output(['git', '-C', '/kaggle/working/Aria', 'rev-parse', '--short', 'HEAD'], text=True).strip()
        print(f'=== Repo HEAD: {head} ===')
    except Exception:
        pass


## 1 · Install dependencies + clone the ARIA repo

~3 minutes. Pinned versions for reproducibility.


In [ ]:
%%bash
set -e
pip install -q --upgrade \
    'transformers>=4.45,<4.50' \
    'trl>=0.13,<0.15' \
    'peft>=0.12,<0.14' \
    'accelerate>=1.0,<2.0' \
    'bitsandbytes>=0.43' \
    'datasets>=3.0' \
    'openenv-core>=0.1' \
    matplotlib

cd /kaggle/working
if [ ! -d Aria ]; then
    git clone --depth 1 https://github.com/Indrajeety993648/Aria.git
else
    cd Aria && git pull --ff-only
fi

cd /kaggle/working/Aria
pip install -q -e backend/packages/aria-contracts
pip install -q -e backend/packages/aria-scenarios
pip install -q -e backend/packages/aria-rewards
pip install -q -e backend/services/env-service
echo '✓ install + clone complete'


## 2 · Smoke-test the environment

Confirms the env can `reset()` + `step()` and the rubric scores correctly.


In [ ]:
import sys, os
os.chdir('/kaggle/working/Aria')

for p in [
    '/kaggle/working/Aria',
    '/kaggle/working/Aria/backend',
    '/kaggle/working/Aria/backend/packages/aria-contracts/src',
    '/kaggle/working/Aria/backend/packages/aria-scenarios/src',
    '/kaggle/working/Aria/backend/packages/aria-rewards/src',
    '/kaggle/working/Aria/backend/services/env-service/src',
]:
    if p not in sys.path:
        sys.path.insert(0, p)

for mod in list(sys.modules):
    if mod.startswith(('aria_contracts', 'aria_scenarios', 'aria_rewards', 'env_service')):
        del sys.modules[mod]

from env_service.aria_env import AriaEnv
from aria_contracts import AriaAction, ActionId

env = AriaEnv()
obs = env.reset(seed=42, category='calendar_conflict', difficulty='medium')
print('rubric tree:')
for name, r in env.rubric.named_rubrics():
    print(f'  {name:<22} weight={r.weight}')

out = env.step(AriaAction(action_id=ActionId.RESOLVE_CONFLICT.value, target_id='conflict_personal'))
print(f'\nstep reward = {out.reward:+.4f}')
print(f'breakdown   = {out.reward_breakdown}')


## 3 · Run A — full reward

Trains GRPO with all 6 reward dimensions active. Time depends on TRAIN_MODE:
  - quick:    ~20 min     (200 steps, 2 generations)
  - detailed: ~50 min     (500 steps, 4 generations)  ← recommended
  - deep:     ~95 min     (1000 steps, 4 generations)

Checkpoints: `/kaggle/working/Aria/checkpoints/aria-full/`


In [ ]:
%%bash
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd /kaggle/working/Aria
mkdir -p checkpoints logs
python -m backend.training.train_grpo \
    --run-name aria-full \
    --steps $ARIA_TRAIN_STEPS \
    --num-generations $ARIA_TRAIN_NGEN \
    --max-completion-len $ARIA_TRAIN_MAXCOMP \
    --max-prompt-len $ARIA_TRAIN_MAXPROMPT \
    --output-dir ./checkpoints/aria-full \
    2>&1 | tee logs/run-a.log
echo '✓ Run A complete'


## 4 · Run B — ablation (`relationship_health` zeroed)

Same training config as Run A — same TRAIN_MODE applies. Identical compute budget makes the
ablation comparison defensible.


In [ ]:
%%bash
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd /kaggle/working/Aria
python -m backend.training.train_grpo \
    --run-name aria-ablate-rh \
    --ablate relationship_health \
    --steps $ARIA_TRAIN_STEPS \
    --num-generations $ARIA_TRAIN_NGEN \
    --max-completion-len $ARIA_TRAIN_MAXCOMP \
    --max-prompt-len $ARIA_TRAIN_MAXPROMPT \
    --output-dir ./checkpoints/aria-ablate-rh \
    2>&1 | tee logs/run-b.log
echo '✓ Run B complete'


## 5 · Plot the two reward curves

Reads `trainer_state.json` from each adapter's checkpoint dir. The fixed plotter walks `final/` and `checkpoint-*/` automatically.


In [ ]:
%%bash
cd /kaggle/working/Aria
python backend/training/plot.py \
    --full-dir ./checkpoints/aria-full \
    --ablate-dir ./checkpoints/aria-ablate-rh \
    --baselines ./backend/baselines/baseline_metrics.json \
    --out-dir ./docs/assets
ls -la docs/assets/


In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/Aria/docs/assets/reward_curve.png'))


## 6 · Evaluate the trained checkpoints (~25 min)

Runs each adapter (full + ablated) over 6 categories × 3 difficulties × 5 seeds = 90 episodes per policy.
Outputs `eval_summary.json` per checkpoint and writes the per-category bar chart.


In [ ]:
%%bash
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd /kaggle/working/Aria
python -m backend.training.eval \
    --model-path ./checkpoints/aria-full/final \
    --out-dir ./eval/full
python -m backend.training.eval \
    --model-path ./checkpoints/aria-ablate-rh/final \
    --ablate relationship_health \
    --out-dir ./eval/ablate
python backend/training/plot.py \
    --full-dir ./checkpoints/aria-full \
    --ablate-dir ./checkpoints/aria-ablate-rh \
    --eval-full ./eval/full/eval_summary.json \
    --eval-ablate ./eval/ablate/eval_summary.json \
    --baselines ./backend/baselines/baseline_metrics.json \
    --out-dir ./docs/assets
ls -la docs/assets/


In [ ]:
from IPython.display import Image, display
import os
for png in ['reward_curve.png', 'category_winrate.png']:
    p = f'/kaggle/working/Aria/docs/assets/{png}'
    if os.path.exists(p):
        print(f'== {png} ==')
        display(Image(p))
    else:
        print(f'(missing) {png}')


## 7 · Download the outputs

All artifacts are in `/kaggle/working/Aria/`. To download:
1. **Save Version** (top-right) → wait for the version to commit
2. Open the version → **Output** tab → download the files you want

Key files for the submission:
- `docs/assets/reward_curve.png` — headline plot
- `docs/assets/category_winrate.png` — eval comparison
- `eval/full/eval_summary.json`, `eval/ablate/eval_summary.json` — raw eval numbers
- `checkpoints/aria-full/final/` and `checkpoints/aria-ablate-rh/final/` — LoRA adapters


In [ ]:
import os, json
print('=== artifacts ready for download ===')
for path in [
    'docs/assets/reward_curve.png',
    'docs/assets/category_winrate.png',
    'eval/full/eval_summary.json',
    'eval/ablate/eval_summary.json',
    'checkpoints/aria-full/final/adapter_config.json',
    'checkpoints/aria-ablate-rh/final/adapter_config.json',
]:
    full = f'/kaggle/working/Aria/{path}'
    size = os.path.getsize(full) if os.path.exists(full) else None
    print(f'  {"✓" if size else "✗"}  {path}  {size or "missing"}')

for kind in ('full', 'ablate'):
    summary_path = f'/kaggle/working/Aria/eval/{kind}/eval_summary.json'
    if os.path.exists(summary_path):
        s = json.loads(open(summary_path).read())
        print(f'\n=== eval/{kind} ===')
        for cell, stats in s.items():
            print(f"  {cell:<32}  mean={stats['mean']:+.3f}  std={stats['std']:.3f}  n={stats['n']}")
